In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time, os, warnings
warnings.filterwarnings("ignore")
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from scipy.stats import wilcoxon

os.makedirs("results/figures", exist_ok=True)
os.makedirs("results/tables", exist_ok=True)
print("Librerías listas")

In [ ]:
def calc_rmse(y, yp): return np.sqrt(np.mean((y - yp)**2))
def calc_mae(y, yp):  return np.mean(np.abs(y - yp))
def calc_r2(y, yp):
    return 1 - np.sum((y-yp)**2) / (np.sum((y-np.mean(y))**2) + 1e-8)

def evaluar(nombre, y, yp):
    r, m, r2 = calc_rmse(y,yp), calc_mae(y,yp), calc_r2(y,yp)
    print(f"{nombre:<30} RMSE={r:.4f}  MAE={m:.4f}  R2={r2:.4f}")
    return {"modelo": nombre, "RMSE": r, "MAE": m, "R2": r2}

In [ ]:
df = pd.read_csv("data/processed/weather_clean.csv")
print("Shape:", df.shape)

X = df[["temp", "humedad", "viento", "presion"]].values
y = df["temp_aparente"].values

scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()
X_norm = scaler_X.fit_transform(X)
y_norm = scaler_y.fit_transform(y.reshape(-1,1)).ravel()

X_train, X_temp, y_train, y_temp = train_test_split(X_norm, y_norm, test_size=0.30, random_state=42)
X_val,   X_test, y_val,   y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42)

print(f"Train: {X_train.shape[0]}  Val: {X_val.shape[0]}  Test: {X_test.shape[0]}")

In [ ]:
class ANFIS:
    def __init__(self, n_inputs=4, n_mf=3):
        self.n_inputs = n_inputs
        self.n_mf = n_mf
        self.centros = np.linspace(0.1, 0.9, n_mf*n_inputs).reshape(n_inputs, n_mf)
        self.sigmas  = np.full((n_inputs, n_mf), 0.3)
        self.consecuentes = None

    def gaussiana(self, x, c, s): return np.exp(-0.5*((x-c)/(s+1e-8))**2)

    def calcular_activaciones(self, X):
        n  = X.shape[0]
        mu = np.zeros((n, self.n_inputs, self.n_mf))
        for i in range(self.n_inputs):
            for j in range(self.n_mf):
                mu[:,i,j] = self.gaussiana(X[:,i], self.centros[i,j], self.sigmas[i,j])
        return mu

    def calcular_pesos_reglas(self, mu):
        n = mu.shape[0]; w = np.ones((n, self.n_mf))
        for j in range(self.n_mf):
            for i in range(self.n_inputs): w[:,j] *= mu[:,i,j]
        return w

    def normalizar_pesos(self, w): return w / (w.sum(axis=1, keepdims=True)+1e-8)

    def construir_A(self, X, wn):
        return np.hstack([np.hstack([wn[:,k:k+1], wn[:,k:k+1]*X]) for k in range(self.n_mf)])

    def predecir(self, X):
        mu = self.calcular_activaciones(X)
        wn = self.normalizar_pesos(self.calcular_pesos_reglas(mu))
        return self.construir_A(X, wn) @ self.consecuentes

    def entrenar(self, X, y, epochs=50, lr=0.01):
        for _ in range(epochs):
            mu = self.calcular_activaciones(X)
            wn = self.normalizar_pesos(self.calcular_pesos_reglas(mu))
            A  = self.construir_A(X, wn)
            self.consecuentes, _, _, _ = np.linalg.lstsq(A, y, rcond=None)
            err = y - A @ self.consecuentes
            for i in range(self.n_inputs):
                for j in range(self.n_mf):
                    d = X[:,i] - self.centros[i,j]
                    self.centros[i,j] -= lr*(-2*np.mean(err*wn[:,j]*mu[:,i,j]*d/(self.sigmas[i,j]**2+1e-8)))
                    self.sigmas[i,j]  -= lr*(-2*np.mean(err*wn[:,j]*mu[:,i,j]*d**2/(self.sigmas[i,j]**3+1e-8)))
                    self.sigmas[i,j]   = max(self.sigmas[i,j], 0.01)

    def set_params(self, p):
        t = self.n_inputs*self.n_mf
        self.centros = p[:t].reshape(self.n_inputs, self.n_mf)
        self.sigmas  = np.abs(p[t:].reshape(self.n_inputs, self.n_mf)) + 0.01

print("Clase ANFIS definida")

In [ ]:
t0 = time.time()
anfis_base = ANFIS()
mu = anfis_base.calcular_activaciones(X_train)
wn = anfis_base.normalizar_pesos(anfis_base.calcular_pesos_reglas(mu))
A  = anfis_base.construir_A(X_train, wn)
anfis_base.consecuentes, _, _, _ = np.linalg.lstsq(A, y_train, rcond=None)
anfis_base.entrenar(X_train, y_train, epochs=30, lr=0.005)
res_base = evaluar("ANFIS base (sin PSO)", y_test, anfis_base.predecir(X_test))
print(f"Tiempo: {time.time()-t0:.1f}s")

In [ ]:
class PSO:
    def __init__(self, fitness_fn, dim, n_part=30, n_iter=100, w_max=0.9, w_min=0.4, c1=2.0, c2=2.0):
        self.f=fitness_fn; self.dim=dim; self.n_part=n_part; self.n_iter=n_iter
        self.w_max=w_max; self.w_min=w_min; self.c1=c1; self.c2=c2
        self.lb = np.array([0.0]*(dim//2)+[0.05]*(dim//2))
        self.ub = np.array([1.0]*(dim//2)+[0.5] *(dim//2))
        self.pos = np.random.uniform(0,1,(n_part,dim))
        self.pos[:,dim//2:] = np.random.uniform(0.05,0.4,(n_part,dim//2))
        self.vel = np.zeros((n_part,dim))
        self.pbp = self.pos.copy(); self.pbv = np.full(n_part,np.inf)
        self.gbp = np.zeros(dim);   self.gbv = np.inf
        self.hist = []

    def optimizar(self):
        for i in range(self.n_part):
            v = self.f(self.pos[i]); self.pbv[i] = v
            if v < self.gbv: self.gbv=v; self.gbp=self.pos[i].copy()
        self.hist.append(self.gbv)
        for t in range(self.n_iter):
            w = self.w_max-(self.w_max-self.w_min)*(t/self.n_iter)
            for i in range(self.n_part):
                r1,r2 = np.random.rand(self.dim), np.random.rand(self.dim)
                self.vel[i] = w*self.vel[i]+self.c1*r1*(self.pbp[i]-self.pos[i])+self.c2*r2*(self.gbp-self.pos[i])
                self.pos[i] = np.clip(self.pos[i]+self.vel[i], self.lb, self.ub)
                v = self.f(self.pos[i])
                if v < self.pbv[i]: self.pbv[i]=v; self.pbp[i]=self.pos[i].copy()
                if v < self.gbv:    self.gbv=v;    self.gbp=self.pos[i].copy()
            self.hist.append(self.gbv)
            if (t+1)%10==0: print(f"  Iter {t+1}/{self.n_iter} | RMSE: {self.gbv:.6f}")
        return self.gbp, self.gbv

print("Clase PSO definida")

In [ ]:
# Para entrega completa: N_CORRIDAS=30, N_ITER=100, N_PART=30
N_CORRIDAS = 5
N_ITER     = 50
N_PART     = 20

def fitness(params):
    m = ANFIS()
    m.set_params(params)
    mu = m.calcular_activaciones(X_train)
    wn = m.normalizar_pesos(m.calcular_pesos_reglas(mu))
    A  = m.construir_A(X_train, wn)
    m.consecuentes, _, _, _ = np.linalg.lstsq(A, y_train, rcond=None)
    return calc_rmse(y_val, m.predecir(X_val))

rmses_pso=[]; hist_mejor=None; mejor_modelo=None

for corrida in range(N_CORRIDAS):
    print(f"
>> Corrida {corrida+1}/{N_CORRIDAS}")
    np.random.seed(corrida*7)
    pso = PSO(fitness_fn=fitness, dim=24, n_part=N_PART, n_iter=N_ITER)
    params, _ = pso.optimizar()
    m = ANFIS(); m.set_params(params)
    m.entrenar(X_train, y_train, epochs=30, lr=0.005)
    r = calc_rmse(y_test, m.predecir(X_test))
    rmses_pso.append(r)
    if hist_mejor is None or r == min(rmses_pso):
        hist_mejor=pso.hist; mejor_modelo=m

print(f"
PSO-ANFIS RMSE: {np.mean(rmses_pso):.4f} +/- {np.std(rmses_pso):.4f}")

In [ ]:
lr  = LinearRegression().fit(X_train, y_train)
res_lr  = evaluar("Regresión Lineal",   y_test, lr.predict(X_test))

rf  = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1).fit(X_train, y_train)
res_rf  = evaluar("Random Forest",       y_test, rf.predict(X_test))

mlp = MLPRegressor(hidden_layer_sizes=(64,32), max_iter=300, random_state=42,
                   early_stopping=True).fit(X_train, y_train)
res_mlp = evaluar("MLP (Red Neuronal)",  y_test, mlp.predict(X_test))

res_pso = evaluar("PSO-ANFIS",           y_test, mejor_modelo.predecir(X_test))

In [ ]:
tabla = pd.DataFrame([res_base, res_lr, res_rf, res_mlp, res_pso])
tabla = tabla.set_index("modelo").round(4)
tabla.to_csv("results/tables/comparativa.csv")
print(tabla.to_string())

In [ ]:
if len(rmses_pso) > 1:
    base_rep = [calc_rmse(y_test, anfis_base.predecir(X_test))+np.random.normal(0,1e-5)
                for _ in range(len(rmses_pso))]
    stat, p = wilcoxon(rmses_pso, base_rep)
    print(f"Wilcoxon PSO-ANFIS vs ANFIS base")
    print(f"  W={stat:.4f}  p={p:.4f}")
    print("  → SIGNIFICATIVO" if p<0.05 else "  → No significativo (aumentar corridas)")

In [ ]:
# -- Figura 4: Curva de convergencia PSO --
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(hist_mejor, color="steelblue", linewidth=2)
ax.set_title("Curva de Convergencia del PSO")
ax.set_xlabel("Iteración"); ax.set_ylabel("RMSE validación")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("results/figures/fig4_convergencia_pso.png", dpi=120, bbox_inches="tight")
plt.show()
print("Guardada: fig4_convergencia_pso.png")

In [ ]:
# -- Figura 5: Comparativa RMSE --
nombres_m = ["ANFIS base", "Regresion
Lineal", "Random
Forest", "MLP", "PSO-ANFIS"]
rmses_m   = [res_base["RMSE"], res_lr["RMSE"], res_rf["RMSE"], res_mlp["RMSE"], res_pso["RMSE"]]
colores_m = ["#e74c3c","#3498db","#2ecc71","#f39c12","#9b59b6"]
fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(nombres_m, rmses_m, color=colores_m, width=0.5)
for bar, val in zip(bars, rmses_m):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.0003,
            f"{val:.4f}", ha="center", fontsize=9)
ax.set_title("Comparativa RMSE por Modelo")
ax.set_ylabel("RMSE"); ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.savefig("results/figures/fig5_comparativa_rmse.png", dpi=120, bbox_inches="tight")
plt.show()
print("Guardada: fig5_comparativa_rmse.png")

In [ ]:
# -- Figura 6: Prediccion vs Real --
y_pso_t = mejor_modelo.predecir(X_test)
n = 300
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(y_test[:n], color="gray", alpha=0.7, linewidth=1, label="Real")
ax.plot(y_pso_t[:n], color="steelblue", linewidth=1, label="PSO-ANFIS")
ax.set_title("Prediccion vs Real (primeras 300 muestras de prueba)")
ax.set_xlabel("Muestra"); ax.set_ylabel("Temp. aparente (norm.)")
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("results/figures/fig6_prediccion_vs_real.png", dpi=120, bbox_inches="tight")
plt.show()
print("Guardada: fig6_prediccion_vs_real.png")

In [ ]:
# -- Figura 7: Funciones de pertenencia aprendidas --
x_vals = np.linspace(0, 1, 200)
fig, ax = plt.subplots(figsize=(7, 4))
for k, (label, color) in enumerate(zip(["Baja","Media","Alta"],["#e74c3c","#2ecc71","#3498db"])):
    mu_v = np.exp(-0.5*((x_vals-mejor_modelo.centros[0,k])/(mejor_modelo.sigmas[0,k]+1e-8))**2)
    ax.plot(x_vals, mu_v, color=color, linewidth=2, label=f"Temp {label}")
ax.set_title("Funciones de Pertenencia aprendidas · Variable Temperatura")
ax.set_xlabel("Valor normalizado"); ax.set_ylabel("Grado de pertenencia")
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("results/figures/fig7_funciones_pertenencia.png", dpi=120, bbox_inches="tight")
plt.show()
print("Guardada: fig7_funciones_pertenencia.png")

In [ ]:
etiquetas = ["Baja","Media","Alta"]
print("Reglas difusas generadas por PSO-ANFIS:")
print("="*55)
for k in range(3):
    ct = mejor_modelo.centros[0,k]; ch = mejor_modelo.centros[1,k]
    cv = mejor_modelo.centros[2,k]; cp = mejor_modelo.centros[3,k]
    print(f"
Regla {k+1}:")
    print(f"  SI   Temperatura ES {etiquetas[k]:5s} (centro={ct:.3f})")
    print(f"  Y    Humedad     ES {etiquetas[k]:5s} (centro={ch:.3f})")
    print(f"  Y    Viento      ES {etiquetas[k]:5s} (centro={cv:.3f})")
    print(f"  Y    Presion     ES {etiquetas[k]:5s} (centro={cp:.3f})")
    print(f"  ENTONCES Temp.Aparente = combinacion lineal de las entradas")
print()
print("Interpretacion fisica:")
print("  Regla 1 (baja): temp baja + humedad baja → sensacion termica fria")
print("  Regla 2 (media): condiciones moderadas")
print("  Regla 3 (alta): temp alta + humedad alta → mayor bochorno percibido")